# 18 · Maschinelles Lernen mit scikit-learn

**Ausblick:** Dieses Notebook ist eine Vorführung ohne Übungen.

**Lernziele**

- Du weißt, was maschinelles Lernen heißt: ein Modell lernt eine Regel aus Beispielen, statt dass du sie von Hand schreibst.
- Du kennst die Begriffe Merkmale `X`, Ziel `y`, Trainings- und Testdaten.
- Du trainierst drei Modelle, die vorhersagen, ob eine Schweißnaht hält, und siehst an Grafiken, wie jedes Modell entscheidet.
- Du sagst mit einer Geraden voraus, wann ein Roboter einen Grenzwert erreicht.
- Du lässt die Landkreise Deutschlands ohne Vorgabe in Gruppen einteilen und zeigst die Gruppen auf einer Karte.

## Was ist scikit-learn?

Bisher hast du Regeln selbst geschrieben: `if strom > 260: ...`. Beim **maschinellen Lernen** schreibst du die Regel nicht, sondern gibst einem Modell Beispiele mit bekanntem Ergebnis. Das Modell sucht selbst eine Regel, die zu den Beispielen passt, und wendet sie auf neue Fälle an.

**scikit-learn** (importiert als `sklearn`) ist die verbreitetste Python-Bibliothek dafür. Sie bietet viele Modelle, Werkzeuge zum Vorbereiten der Daten und zum Prüfen der Ergebnisse. Sie arbeitet direkt mit pandas-Tabellen aus Notebook 15.

Die Daten bestehen immer aus zwei Teilen: den **Merkmalen** `X`, also dem, was wir messen, und dem **Ziel** `y`, also dem, was das Modell vorhersagen soll.

![Eine Tabelle mit sechs Schweißnähten. Die Spalten Geschwindigkeit_cm_min und Strom_A sind blau und als Merkmale X beschriftet (was wir messen), die Spalte Ergebnis mit ok und nicht ok ist grün und als Ziel y beschriftet (was wir vorhersagen). Darunter: Eine Zeile je Beispiel, das Modell lernt, wie y von X abhängt. Groß X ist eine Tabelle mit mehreren Spalten, klein y eine einzelne Spalte.](../assets/18_x_und_y.png)

Jedes Modell in scikit-learn hat dieselben zwei Methoden: `fit()` lernt aus Beispielen, `predict()` sagt für neue Fälle das Ergebnis voraus.

![Oben: Beispiele mit Ergebnis (X: Geschwindigkeit, Strom; y: ok oder nicht ok) gehen mit model.fit(X, y) in das Modell, das aus den Beispielen eine Regel gelernt hat. Unten: neue Nähte, nur X, Ergebnis unbekannt, gehen mit model.predict(X_neu) in dasselbe Modell, heraus kommt eine Vorhersage ok oder nicht ok je neue Naht.](../assets/18_fit_predict.png)

In diesem Notebook lernst du zwei Arten kennen:

| Art | Daten | Frage | Beispiel hier |
| --- | --- | --- | --- |
| überwachtes Lernen | `X` und `y` | Welches Ergebnis hat ein neuer Fall? | Hält die Naht? Wann ist der Grenzwert erreicht? |
| unüberwachtes Lernen | nur `X` | Welche Fälle ähneln sich? | Welche Landkreise bilden Gruppen? |

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

### Ein erstes Modell mit sechs Nähten

Die sechs Nähte aus der Grafik als kleine Tabelle. Mit doppelten eckigen Klammern wählst du mehrere Spalten als `X`, mit einfachen eine Spalte als `y`, wie in Notebook 15:

In [ ]:
naehte = pd.DataFrame({
    "Geschwindigkeit_cm_min": [45, 38, 52, 47, 35, 50],
    "Strom_A": [215, 260, 180, 225, 240, 230],
    "Ergebnis": ["ok", "nicht ok", "nicht ok", "ok", "nicht ok", "ok"],
})
X_mini = naehte[["Geschwindigkeit_cm_min", "Strom_A"]]
y_mini = naehte["Ergebnis"]
naehte

Als Modell nehmen wir die **nächsten Nachbarn** (`KNeighborsClassifier`). Die Idee ist einfach: Für eine neue Naht sucht das Modell die ähnlichsten bekannten Nähte, also die mit den nächstgelegenen Werten, und übernimmt das Ergebnis, das unter ihnen am häufigsten vorkommt. `n_neighbors=3` legt fest, wie viele Nachbarn abstimmen.

Die drei Schritte:

1. Modell anlegen: Noch hat es nichts gelernt.
2. `fit(X_mini, y_mini)`: Das Modell merkt sich die Beispiele.
3. `predict(neu)`: Vorhersage für zwei neue Nähte. `neu` braucht dieselben Spalten wie `X_mini`, aber kein Ergebnis.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_mini, y_mini)

neu = pd.DataFrame({"Geschwindigkeit_cm_min": [44, 36], "Strom_A": [220, 255]})
model.predict(neu)

Die erste neue Naht (44 cm/min, 220 A) liegt nahe bei den guten Nähten mit 45/215 und 47/225: `ok`. Die zweite (36 cm/min, 255 A) liegt nahe bei 38/260 und 35/240, beide nicht ok: `nicht ok`.

Sechs Beispiele sind natürlich zu wenig, um einem Modell zu trauen. Wie gut ein Modell ist, prüft man an Fällen, die es beim Lernen **nicht** gesehen hat. Deshalb teilt man die Daten vorher auf:

![train_test_split(X, y, test_size=0.25): 600 Nähte mit bekanntem Ergebnis, zufällig gemischt, werden in Training (450 Nähte, 75 %) und Test (150 Nähte, 25 %) geteilt. Mit model.fit(X_train, y_train) lernt das Modell nur aus den Trainingsnähten; model.score(X_test, y_test) gibt den Anteil richtig vorhergesagter Nähte an, die es nie gesehen hat. Darunter: Wie eine Prüfung mit neuen Aufgaben: Wer die Übungsaufgaben nur auswendig gelernt hat, fällt hier auf.](../assets/18_train_test.png)

## Klassifikation: Hält die Schweißnaht?

Ein Modell, das eine Kategorie vorhersagt (hier `ok` oder `nicht ok`), heißt **Klassifikator**. Wir nehmen das Schweißprotokoll aus Notebook 15: 600 Nähte, für jede Strom, Geschwindigkeit und das Prüfergebnis. Die Werte sind simuliert, erzeugt mit `tools/make_ausblick_data.py`.

In [ ]:
log = pd.read_csv(
    "../daten/schweissprotokoll.csv",
    sep=";",
    decimal=",",
    parse_dates=["Zeitpunkt"],
    date_format="%d.%m.%Y %H:%M",
)
log.head()

Das Streudiagramm aus Notebook 15 zeigt, was das Modell lernen soll. `colors` übersetzt jedes Ergebnis in eine Farbe; `alpha=0.6` macht die Punkte etwas durchsichtig, damit man übereinanderliegende erkennt:

In [ ]:
colors = {"ok": "green", "nicht ok": "darkred"}

log.plot.scatter(
    x="Geschwindigkeit_cm_min", y="Strom_A", c=log["Ergebnis"].map(colors), alpha=0.5, figsize=(7, 5)
)
plt.show()

Die guten Nähte (grün) liegen in einem schrägen Band von links unten nach rechts oben: Wer schneller fährt, braucht mehr Strom. Links oben und ganz links (viel Strom für eine langsame Fahrt: brennt durch) und rechts unten (zu wenig Strom für eine schnelle Fahrt: hält nicht) liegen die schlechten. Ein paar Punkte passen nicht ins Muster: Auch in der Praxis hält mal eine Naht nicht, obwohl die Werte stimmen.

Jetzt teilen wir in Merkmale und Ziel und dann in Training und Test. Die Argumente von `train_test_split()`:

- `X, y`: die Daten; beide werden zeilengleich aufgeteilt, damit jede Naht ihr Ergebnis behält.
- `test_size=0.25`: ein Viertel der Nähte kommt in den Test.
- `random_state=0`: Die Aufteilung ist zufällig; mit einer festen Zahl kommt bei jedem Lauf dieselbe heraus, so passen die Zahlen im Text zu deinen Ausgaben.
- `stratify=y`: In Training und Test ist der Anteil schlechter Nähte gleich groß wie in allen Daten.

Die Funktion gibt vier Teile zurück, die wir in vier Variablen auspacken:

In [ ]:
X = log[["Geschwindigkeit_cm_min", "Strom_A"]]
y = log["Ergebnis"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(len(X_train), "Nähte zum Lernen,", len(X_test), "zum Prüfen")
y_test.value_counts()

Bevor wir ein Modell loben, brauchen wir einen Vergleich. Der einfachste „Klassifikator“ sagt immer `ok`. Wie oft hätte er im Test recht? `y_test == "ok"` ist eine Maske aus `True` und `False`; ihr Mittelwert ist der Anteil der `True`:

In [ ]:
(y_test == "ok").mean()

Wer immer `ok` sagt, hat in 68 % der Fälle recht, ohne irgendetwas gelernt zu haben. Ein Modell muss deutlich besser sein.

Um zu sehen, wie ein Modell entscheidet, färben wir den ganzen Bereich des Diagramms danach ein, was das Modell an jeder Stelle vorhersagen würde: grün für `ok`, rot für `nicht ok`. Die Linie zwischen den Farben heißt **Entscheidungsgrenze**. Darauf zeichnen wir die Testnähte mit ihrem echten Ergebnis. Weil wir das für jedes Modell brauchen, kommt es in eine Funktion (Notebook 07):

- `DecisionBoundaryDisplay.from_estimator()` fragt das Modell an vielen Punkten im Bereich von `X` nach seiner Vorhersage (`response_method="predict"`) und färbt die Fläche mit den Farben aus `cmap`; `"RdYlGn"` reicht von Rot über Gelb nach Grün. `alpha=0.3` hält die Fläche blass, `ax=ax` zeichnet in unsere Grafik.
- `ax.scatter()` zeichnet die Testnähte darüber, mit schwarzem Rand (`edgecolors`), damit sie sich von der Fläche abheben.
- `model.score(X_test, y_test)` ist der Anteil richtiger Vorhersagen im Test; er kommt in den Titel.

In [ ]:
def show_boundary(model, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    DecisionBoundaryDisplay.from_estimator(
        model, X, response_method="predict", cmap="RdYlGn", alpha=0.3, ax=ax
    )
    ax.scatter(
        X_test["Geschwindigkeit_cm_min"], X_test["Strom_A"], c=y_test.map(colors), edgecolors="black"
    )
    ax.set_title(f"{title}: {model.score(X_test, y_test):.0%} richtig im Test")
    plt.show()

### Modell 1: eine gerade Grenze

Die **logistische Regression** (`LogisticRegression`) trennt die beiden Gruppen mit einer geraden Linie. Trotz des Namens ist sie ein Klassifikator. Sie ist schnell und leicht zu erklären und deshalb oft das erste Modell, das man ausprobiert:

In [ ]:
linear = LogisticRegression()
linear.fit(X_train, y_train)
show_boundary(linear, "Logistische Regression")

72 % richtig, kaum besser als „immer ok“. Die Grafik zeigt den Grund: Die Gerade schneidet nur die Ecke links oben ab. Das Band der guten Nähte hat aber zwei Ränder, und die schlechten Nähte rechts unten sagt das Modell alle als `ok` vorher. Eine einzige Gerade kann zwei Ränder nicht nachzeichnen. Das Modell ist nicht falsch programmiert, es passt nur nicht zu den Daten. Welches Modell passt, sieht man oft erst an solchen Grafiken.

### Modell 2: ein Entscheidungsbaum

Ein **Entscheidungsbaum** (`DecisionTreeClassifier`) lernt eine Folge von Ja-Nein-Fragen, so wie man sie selbst aufschreiben würde: „Ist der Strom höher als …? Wenn ja: Ist die Geschwindigkeit kleiner als …?“ Jede Frage teilt die Nähte so, dass in den Teilen möglichst nur gute oder nur schlechte übrig bleiben.

- `max_depth=3`: höchstens drei Fragen hintereinander. Warum man die Tiefe begrenzt, zeigen wir gleich.
- `random_state=0`: Bei gleich guten Fragen wählt der Baum zufällig; so bleibt das Ergebnis bei jedem Lauf gleich.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
show_boundary(tree, "Entscheidungsbaum, Tiefe 3")

80 % richtig. Jede Frage des Baums vergleicht eine einzige Spalte mit einem Wert, deshalb besteht die Grenze nur aus waagerechten und senkrechten Stücken: Der Baum baut das schräge Band aus Rechtecken nach, wie eine Treppe. Mit drei Fragen gelingt das nur grob. Der schmale grüne Streifen ganz links beruht auf einer einzigen Trainingsnaht, wie der Baum gleich zeigt.

Das Schöne am Baum: Man kann ihn lesen. `plot_tree()` zeichnet die gelernten Fragen:

- `feature_names` und `class_names` setzen die Spaltennamen und Ergebnisse ein, sonst stünde dort nur `x[0]` und `y[1]`.
- `filled=True` färbt jeden Kasten nach der Mehrheit darin: orange für `nicht ok`, blau für `ok`; je kräftiger, desto eindeutiger.
- `impurity=False` blendet eine Kennzahl aus, die wir hier nicht brauchen.
- `fontsize=10` hält die Schrift lesbar; `figsize` macht Platz für die acht Kästen der untersten Reihe.

So liest du einen Kasten: oben die Frage, darunter `samples` (wie viele Trainingsnähte hier ankommen), `value` (wie viele davon `nicht ok` und `ok` sind) und `class` (was der Baum hier vorhersagt). Ist die Antwort auf die Frage Ja (`True`), geht es nach links weiter, sonst nach rechts.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    tree, feature_names=X.columns, class_names=tree.classes_, filled=True, impurity=False, fontsize=10, ax=ax
)
plt.show()

Die erste Frage trennt nach der Geschwindigkeit: Bei höchstens 38,15 cm/min (links) sind 50 der 71 Nähte nicht ok, langsame Fahrt ist riskant. Rechts davon fragt der Baum, ob die Fahrt auch nicht zu schnell ist (bis 55,45 cm/min), und dann nach dem Strom. Der größte Kasten unten zählt 300 Trainingsnähte, 257 davon ok: das Kernstück des guten Bands. In der untersten Reihe steht auch ein Kasten mit `samples = 1`: Diese eine Naht hat den grünen Streifen ganz links erzeugt.

Solche Regeln könnte man auch einem Menschen an der Anlage geben. Das ist der große Vorteil eines Baums gegenüber anderen Modellen.

#### Wie tief darf der Baum werden?

Mit mehr Fragen könnte der Baum das Band feiner nachzeichnen. Ohne Grenze fragt er so lange, bis jede Trainingsnaht richtig eingeordnet ist, auch die Ausreißer. Dann hat er die Trainingsdaten **auswendig gelernt**, und das hilft bei neuen Nähten nicht. Die Schleife trainiert Bäume mit 1 bis 12 Fragen und notiert für jeden den Anteil richtiger Vorhersagen im Training und im Test:

In [ ]:
rows = []
for depth in range(1, 13):
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    rows.append({
        "Tiefe": depth,
        "Training": candidate.score(X_train, y_train),
        "Test": candidate.score(X_test, y_test),
    })
depths = pd.DataFrame(rows).set_index("Tiefe")

ax = depths.plot(marker="o", figsize=(7, 4), ylabel="Anteil richtig")
ax.set_xticks(depths.index)
plt.show()

Im Training (blau) wird der Baum mit jeder Frage besser und erreicht fast 100 %. Im Test (orange) steigt er bis Tiefe 6 auf 86 % und kommt danach nicht mehr weiter: Die zusätzlichen Fragen lernen Ausreißer der Trainingsdaten, die bei neuen Nähten nicht wiederkommen. Die Lücke zwischen den Linien zeigt, wie viel der Baum auswendig gelernt hat. Ein Baum mit Tiefe 6 träfe also etwas besser als unser Baum mit Tiefe 3; wir haben 3 gewählt, weil man ihn noch lesen kann.

Deshalb prüft man immer am Test und nicht am Training: Das Training sagt nur, wie gut das Modell auswendig gelernt hat.

### Modell 3: die nächsten Nachbarn

Zurück zum Modell vom Anfang: `KNeighborsClassifier` sieht sich für jede Naht die ähnlichsten Trainingsnähte an. Hier stimmen 15 Nachbarn ab (`n_neighbors=15`); mit mehr Nachbarn fallen einzelne Ausreißer weniger ins Gewicht.

Vorher müssen wir eine Falle umgehen. „Ähnlich“ heißt für das Modell: geringer Abstand in Zahlen. Ein Unterschied von 10 A beim Strom zählt dann genauso viel wie 10 cm/min bei der Geschwindigkeit. Der Strom schwankt aber viel stärker: `log.describe()` zeigt als Standardabweichung (`std`) rund 28 A gegen 7 cm/min. Er würde den Abstand fast allein bestimmen.

Der `StandardScaler` rechnet deshalb jede Spalte um: in „wie viele Standardabweichungen über oder unter dem Mittelwert“. Danach zählen beide Spalten gleich viel. `make_pipeline()` hängt Umrechnen und Modell zu einem Objekt zusammen: `fit()` lernt dann beides aus den Trainingsdaten, `predict()` rechnet neue Nähte zuerst genauso um. Für alles Weitere ist die Pipeline einfach ein Modell:

In [ ]:
neighbors = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
neighbors.fit(X_train, y_train)
show_boundary(neighbors, "15 nächste Nachbarn")

91 % richtig. Die Grenze ist weder gerade noch eckig, sie folgt dem Band mit beiden Rändern. Das Modell kennt keine Formel, es vergleicht nur mit bekannten Nähten; deshalb kann seine Grenze jede Form annehmen.

Besser geht es kaum: Die Simulation lässt auch Nähte mit guten Werten in 3 % der Fälle scheitern und Nähte mit schlechten Werten in 15 % der Fälle halten. Selbst die Regel, mit der die Daten erzeugt wurden, trifft im Test nur 95 %. Zufall kann kein Modell vorhersagen.

### Welche Fehler macht das Modell?

Der Anteil richtiger Vorhersagen sagt nicht, **welche** Fehler das Modell macht. Bei Schweißnähten ist das wichtig: Eine gute Naht verwerfen kostet ein Bauteil. Eine schlechte Naht durchlassen kann ein Bauteil im Einsatz brechen lassen.

Die **Konfusionsmatrix** zählt beides auf: In den Zeilen steht das echte Ergebnis (`True label`), in den Spalten die Vorhersage (`Predicted label`). Auf der Diagonalen stehen die Treffer, daneben die Fehler. `from_estimator()` lässt das Modell die Testnähte vorhersagen und zählt; `cmap="Blues"` färbt große Zahlen dunkler:

In [ ]:
ConfusionMatrixDisplay.from_estimator(neighbors, X_test, y_test, cmap="Blues")
plt.show()

Von den 48 schlechten Nähten im Test erkennt das Modell 37; 11 würde es als `ok` durchlassen. Von den 102 guten verwirft es nur 3. Für eine Qualitätsprüfung sind die 11 durchgelassenen Nähte der wichtigere Fehler. Wer ihn verringern will, nimmt in Kauf, dass mehr gute Nähte verworfen werden; wie man das einstellt, geht über diesen Ausblick hinaus.

Zum Schluss der Zweck des Ganzen: Vorhersagen für Nähte, die gerade erst geplant werden.

In [ ]:
geplant = pd.DataFrame({"Geschwindigkeit_cm_min": [45, 35, 55], "Strom_A": [215, 260, 190]})
geplant["Vorhersage"] = neighbors.predict(geplant)
geplant

Die erste Naht hat übliche Werte und hält laut Modell. Die zweite fährt langsam mit viel Strom (brennt durch), die dritte schnell mit wenig Strom (hält nicht). So kann man Einstellungen prüfen, bevor die erste Naht geschweißt ist.

## Regression: Wann muss R3 gewartet werden?

Ein Modell, das eine Zahl vorhersagt statt einer Kategorie, heißt **Regression**. In Notebook 15 fiel auf, dass Roboter R3 mit der Zeit immer mehr Strom zieht. Wenn das so weitergeht: Wann erreicht er einen Grenzwert, ab dem er gewartet werden muss?

Als Merkmal brauchen wir die Zeit als Zahl. Wir rechnen jeden Zeitpunkt in Tage seit der ersten Naht von R3 um: Die Differenz zweier Zeitpunkte ist eine Zeitspanne (Notebook 11), geteilt durch einen Tag (`pd.Timedelta(days=1)`) wird daraus eine Kommazahl. `copy()` macht aus der Auswahl eine eigene Tabelle, damit die neue Spalte das Protokoll nicht verändert:

In [ ]:
r3 = log[log["Roboter"] == "R3"].copy()
start = r3["Zeitpunkt"].min()
r3["Tag"] = (r3["Zeitpunkt"] - start) / pd.Timedelta(days=1)

print(len(r3), "Nähte von R3 ab", start)
r3.plot.scatter(x="Tag", y="Strom_A", alpha=0.6, figsize=(7, 4))
plt.show()

Die einzelnen Nähte streuen stark, aber der Trend steigt. Die **lineare Regression** (`LinearRegression`) legt die Gerade durch die Punktwolke, die am besten passt: die, bei der die Punkte im Mittel am wenigsten von ihr abweichen. Gelernt wird wie bei den Klassifikatoren mit `fit(X, y)`; `X` ist die Tabelle mit der Spalte `Tag` (doppelte Klammern), `y` der Strom.

Nach dem Lernen stehen die zwei Zahlen der Geraden im Modell: `coef_` ist die Steigung (Ampere mehr pro Tag; ein Wert je Merkmal, deshalb `[0]`), `intercept_` der Wert am Tag 0. Der Unterstrich am Ende zeigt in scikit-learn: Dieser Wert wurde gelernt.

In [ ]:
trend = LinearRegression()
trend.fit(r3[["Tag"]], r3["Strom_A"])

slope = trend.coef_[0]
print(f"Start: {trend.intercept_:.1f} A, Anstieg: {slope:.2f} A pro Tag")

Nehmen wir an, ab einem mittleren Strom von 300 A muss R3 gewartet werden (eine angenommene Grenze für dieses Beispiel). Auf der Geraden gilt `Strom = intercept_ + slope * Tag`; nach dem Tag aufgelöst ergibt das den Tag, an dem die Grenze erreicht ist. Plus `start` wird daraus ein Datum.

Die Grafik zeigt die Messwerte, die Gerade über die Messdaten hinaus verlängert und die Grenze:

- `trend.predict()` berechnet die Gerade für die Tage 0 bis 35; `days` ist eine Tabelle mit der Spalte `Tag`, wie beim Lernen.
- `axhline()` zeichnet eine waagerechte Linie (die Grenze), `axvline()` eine senkrechte (den Tag), `linestyle="--"` gestrichelt.
- `label` beschriftet jede Linie für die Legende; `legend(loc="lower right")` setzt die Legende nach rechts unten, wo keine Punkte liegen.

In [ ]:
limit = 300
limit_day = (limit - trend.intercept_) / slope
print(f"Grenze nach {limit_day:.1f} Tagen erreicht, am {start + pd.Timedelta(days=limit_day):%d.%m.%Y}")

days = pd.DataFrame({"Tag": [0, 35]})
ax = r3.plot.scatter(x="Tag", y="Strom_A", alpha=0.4, figsize=(9, 5), label="Nähte von R3")
ax.plot(days["Tag"], trend.predict(days), color="black", label="gelernte Gerade")
ax.axhline(limit, color="red", linestyle="--", label=f"Grenze {limit} A")
ax.axvline(limit_day, color="grey", linestyle="--", label=f"Tag {limit_day:.0f}")
ax.legend(loc="lower right")
plt.show()

Die Gerade beginnt bei rund 215 A und steigt um knapp 2,8 A pro Tag. Gemessen wurde bis Tag 21; die Grenze von 300 A erreicht der Trend nach knapp 31 Tagen, am 2. Oktober. Einzelne Nähte liegen schon heute über 300 A: Die Grenze gilt hier für den Trend, also für den mittleren Strom, nicht für jede einzelne Naht.

**Vorsicht beim Verlängern:** Die Gerade nimmt an, dass der Strom genauso weiter steigt wie in den gemessenen drei Wochen. Ob das stimmt, weiß das Modell nicht. Je weiter die Vorhersage über die Messdaten hinausgeht, desto mehr ist sie eine Annahme. In der Praxis rechnet man nach jeder Woche mit den neuen Messwerten neu.

## Unüberwacht: Welche Landkreise ähneln sich?

Bisher hatte jedes Beispiel ein bekanntes Ergebnis `y`. Beim **unüberwachten Lernen** gibt es keins: Das Modell soll selbst Gruppen finden, in denen sich die Fälle ähneln. Das Verfahren **k-Means** teilt die Daten in `k` Gruppen (*Cluster*) ein; die Zahl `k` legst du fest.

So arbeitet k-Means: Es setzt `k` Mittelpunkte, ordnet jeden Fall dem nächsten Mittelpunkt zu, schiebt jeden Mittelpunkt in die Mitte seiner Fälle und wiederholt das, bis sich nichts mehr ändert.

Als Fälle nehmen wir die 400 Landkreise aus Notebook 17, mit Umrissen, Einwohnern und Dichte. Woher die Daten kommen, steht dort und am Ende dieses Notebooks.

In [ ]:
kreise = gpd.read_file("../daten/kreise.gpkg")
einwohner = pd.read_csv(
    "../daten/kreise_bevoelkerung.csv", sep=";", decimal=",", dtype={"Schluessel": str}
)
karte = kreise.merge(einwohner[["Schluessel", "Bundesland", "Flaeche_km2", "Einwohner"]], on="Schluessel")
karte["Dichte"] = (karte["Einwohner"] / karte["Flaeche_km2"]).round()
karte[["Einwohner", "Dichte"]].describe().round()

Ein erster Versuch mit den Spalten, wie sie sind. `KMeans(n_clusters=3)` sucht drei Gruppen. Wo die Mittelpunkte am Anfang stehen, wird zufällig gewählt, und das Ergebnis hängt davon ab. Zwei Argumente regeln das:

- `n_init=10`: k-Means startet zehnmal, jedes Mal von anderen zufälligen Startpunkten, und behält das beste der zehn Ergebnisse.
- `random_state=0`: legt den Zufallsgenerator fest, aus dem die zehn Starts nacheinander gezogen werden. Die zehn Starts bleiben untereinander verschieden, aber bei jedem Lauf sind es dieselben zehn. So passen die Zahlen im Text zu deinen Ausgaben, wie bei `train_test_split()`.

`fit_predict()` lernt und gibt gleich für jeden Kreis die Nummer seiner Gruppe zurück; es gibt ja kein `y`, nur `X`:

In [ ]:
features = karte[["Dichte", "Einwohner"]]
raw_groups = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(features)
karte.groupby(raw_groups)["Einwohner"].agg(["count", "min", "max"])

Das Ergebnis ist unbrauchbar: 369 Kreise in einer Gruppe, 30 in der zweiten, Berlin allein in der dritten. Die Gruppen trennen nur nach Einwohnern (bis 395 000, ab 408 000, Berlin). Die Dichte spielt keine Rolle, denn Einwohner zählen in Hunderttausenden, die Dichte nur in Hunderten. Im Abstand zählt ein Einwohner mehr oder weniger so viel wie ein Einwohner je km² mehr oder weniger, also bestimmen die großen Zahlen alles. Es ist dieselbe Falle wie beim Strom.

Zwei Schritte helfen:

1. **Logarithmus:** `np.log10()` zählt Zehnerstufen: 100 wird zu 2, 1000 zu 3, eine Million zu 6. Berlin hat rund 18-mal so viele Einwohner wie ein Kreis im Mittel; logarithmiert ist das ein Abstand von nur 1,25 Stufen. Wenige riesige Werte drücken dann nicht mehr alle anderen zusammen, wie bei der Farbskala in Notebook 17.
2. **Skalieren** mit dem `StandardScaler` wie bei den Nachbarn, damit Dichte und Einwohner gleich viel zählen.

Die Grafik vergleicht beide Versuche. Beide Achsen sind logarithmisch (`set_xscale("log")`, `set_yscale("log")`), damit man die Kreise überhaupt auseinanderhält; `sharey=True` gibt beiden Diagrammen dieselbe senkrechte Achse. Die Farbe (`c`) ist die Gruppennummer, `cmap="brg"` färbt die Nummern 0, 1 und 2 blau, rot und grün:

In [ ]:
clustering = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=0))
groups = clustering.fit_predict(np.log10(features))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, labels, title in [
    (axes[0], raw_groups, "Werte, wie sie sind"),
    (axes[1], groups, "logarithmiert und skaliert"),
]:
    ax.scatter(karte["Einwohner"], karte["Dichte"], c=labels, cmap="brg", alpha=0.7)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Einwohner")
    ax.set_title(title)
axes[0].set_ylabel("Einwohner je km²")
plt.show()

Links teilt k-Means die Kreise mit senkrechten Schnitten nach der Einwohnerzahl, die Dichte (senkrechte Achse) spielt keine Rolle. Rechts entstehen drei Gruppen, die man mit bloßem Auge erkennt: unten die dünn besiedelten Kreise (blau), oben links dicht besiedelte Kreise mit wenigen Einwohnern (grün), oben rechts die mit vielen Einwohnern (rot).

Was die Gruppen bedeuten, sagt k-Means nicht; das finden wir heraus. Die Nummer jedes Kreises kommt als Spalte in die Tabelle, `groupby()` zeigt den Median jeder Gruppe:

In [ ]:
karte["Gruppe_Nr"] = groups
karte.groupby("Gruppe_Nr")[["Einwohner", "Dichte"]].median()

- Gruppe 0: rund 134 000 Einwohner und 121 je km², also dünn besiedelt: **ländlich**.
- Gruppe 1: rund 312 000 Einwohner und 640 je km²: **groß**, die Großstädte und einwohnerstarke Kreise.
- Gruppe 2: nur rund 91 000 Einwohner, aber 1064 je km², viele Menschen auf kleiner Fläche: **kleinere Stadt**.

Die Nummern hängen von `random_state` ab; mit einer anderen Zahl können dieselben Gruppen andere Nummern bekommen. Deshalb vergeben wir die Namen erst nach dem Blick auf die Tabelle, mit einem Dict und `map()` wie in Notebook 15.

Passen die Namen? Die Tabelle aus Notebook 17 kennt für jeden Kreis seine Art. k-Means hat diese Spalte nie gesehen. `pd.crosstab()` zählt, wie oft jede Gruppe mit jeder Art zusammen vorkommt:

In [ ]:
names = {0: "ländlich", 1: "groß", 2: "kleinere Stadt"}
karte["Gruppe"] = karte["Gruppe_Nr"].map(names)
pd.crosstab(karte["Gruppe"], karte["Art"])

Die Gruppe „ländlich“ besteht nur aus Landkreisen und Kreisen (223), „kleinere Stadt“ fast nur aus kreisfreien Städten und Stadtkreisen (65 von 68). „Groß“ mischt beides: Großstädte und Landkreise mit vielen Einwohnern. Ohne die Spalte `Art` zu kennen, hat k-Means die Einteilung in Stadt und Land fast nachgebaut und eine dritte Gruppe dazugefunden.

Jetzt die Karte, wie in Notebook 17. Neu ist `categorical=True`: Die Spalte `Gruppe` enthält Namen statt Zahlen, jeder Name bekommt eine eigene Farbe aus `cmap="Set2"`. Die Ländergrenzen entstehen wieder mit `dissolve()` aus den Kreisen:

In [ ]:
laender = karte.dissolve(by="Bundesland")

ax = karte.plot(
    column="Gruppe",
    categorical=True,
    cmap="Set2",
    edgecolor="white",
    linewidth=0.2,
    figsize=(12, 14),
    legend=True,
    legend_kwds={"title": "Gruppe (k-Means)", "loc": "upper left", "bbox_to_anchor": (1, 1)},
)
laender.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_axis_off()

Die großen Gruppen werden auf der Karte zu Regionen: Fast ganz Nordrhein-Westfalen, das Rhein-Main-Gebiet, der Raum Stuttgart und das Umland von Hamburg sind „groß“, ebenso Berlin, Hamburg und München. Die kleineren Städte liegen als einzelne Flecken verstreut, besonders viele in Bayern, das viele kreisfreie Städte hat. Der Rest ist ländlich.

Drei Dinge gelten für k-Means immer:

- Die Zahl der Gruppen legst du fest. Mit `n_clusters=4` entstehen vier Gruppen, und keine Antwort ist „die richtige“. Probier es aus.
- Die Nummern der Gruppen bedeuten nichts; die Namen vergibst du, nachdem du dir die Gruppen angesehen hast.
- Das Ergebnis hängt davon ab, welche Spalten du gibst und wie du sie vorbereitest, wie der erste Versuch gezeigt hat.

## Zusammenfassung

- Maschinelles Lernen heißt: Ein Modell lernt eine Regel aus Beispielen. Die Merkmale heißen `X` (Tabelle), das Ziel `y` (Spalte).
- Jedes Modell in scikit-learn hat `fit()` zum Lernen und `predict()` zum Vorhersagen; `score()` misst den Anteil richtiger Vorhersagen.
- `train_test_split()` hält Testdaten zurück. Geprüft wird nur an Daten, die das Modell beim Lernen nicht gesehen hat; das Training allein belohnt Auswendiglernen.
- Vergleiche jedes Modell mit dem einfachsten Raten (hier: immer „ok“).
- Die Entscheidungsgrenze zeigt, wie ein Klassifikator denkt: gerade bei der logistischen Regression, in Stufen beim Entscheidungsbaum, frei geformt bei den nächsten Nachbarn. Den Baum kann man mit `plot_tree()` sogar lesen.
- Die Konfusionsmatrix zeigt, welche Fehler ein Modell macht.
- Modelle, die mit Abständen rechnen, brauchen Spalten auf gleicher Skala: `StandardScaler` in einer `make_pipeline()`.
- `LinearRegression` sagt Zahlen voraus; `coef_` und `intercept_` sind die gelernte Gerade. Vorhersagen über die Messdaten hinaus sind Annahmen.
- `KMeans` findet Gruppen ohne `y`; die Zahl der Gruppen und ihre Namen legst du fest.

Wie geht es weiter? Mehr Modelle und Beispiele zeigt das Handbuch von scikit-learn: https://scikit-learn.org/stable/user_guide.html. Die nächsten Begriffe dort sind Kreuzvalidierung (mehrmals mit wechselnden Testdaten prüfen) und Pipelines mit mehreren Schritten.

**Quellen der Daten:**

- Schweißprotokoll: simuliert, erzeugt mit `tools/make_ausblick_data.py`.
- Umrisse der Kreise: © BKG (2026) dl-de/by-2-0 (Daten verändert), Verwaltungsgebiete 1:2 500 000 (VG2500), Stand 31.12.2024; Datenquellen: https://sgx.geodatenzentrum.de/web_public/gdz/datenquellen/datenquellen_vg_nuts.pdf
- Fläche und Einwohner: © Statistisches Bundesamt (Destatis), 2025, Gemeindeverzeichnis, Stand 31.12.2024.